In [0]:
from pyspark.sql.functions import col, from_json, expr, current_date, date_sub
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Filter for records created in the last 7 days (for weekly incremental processing)
raw_df = spark.read.table("workspace.fotmob.raw_sofascore_player_overview").filter(
    col("createdate") >= date_sub(current_date(), 7)
)

print(f"Processing {raw_df.count()} raw records from the last 7 days...")

# Define schema for sofascore player overview JSON
position_detail_schema = StructType([
    StructField("name", StringType(), True),
    StructField("shortName", StringType(), True)
])

schema = StructType([
    StructField("player", StructType([
        StructField("name", StringType(), True),
        StructField("firstName", StringType(), True),
        StructField("lastName", StringType(), True),
        StructField("dateOfBirthTimestamp", IntegerType(), True),
        StructField("preferredFoot", StringType(), True),
        StructField("gender", StringType(), True),
        StructField("country", StructType([
            StructField("name", StringType(), True),
            StructField("alpha2", StringType(), True)
        ]), True),
        StructField("position", StringType(), True),
        StructField("positionsDetailed", ArrayType(position_detail_schema), True),
        StructField("team", StructType([
            StructField("name", StringType(), True),
            StructField("id", IntegerType(), True)
        ]), True)
    ]), True)
])

# Parse JSON and extract fields
df = raw_df.select(
    col("player_id"),
    col("createdate"),
    from_json(col("raw_json"), schema).alias("parsed")
).filter(
    col("parsed.player.gender") == "F"  # Only process female players
).select(
    col("player_id"),
    col("createdate"),
    col("parsed.player.name").alias("player_name"),
    expr("from_unixtime(parsed.player.dateOfBirthTimestamp)").cast("date").alias("birthdate"),
    col("parsed.player.team.id").alias("club_id"),
    col("parsed.player.team.name").alias("club"),
    col("parsed.player.position").alias("primary_position"),
    expr("transform(parsed.player.positionsDetailed, pos -> pos.name)").alias("secondary_positions"),
    col("parsed.player.preferredFoot").alias("preferred_foot"),
    col("parsed.player.country.name").alias("country")
)

print(f"After filtering for female players: {df.count()} records")
display(df)

Processing 493 raw records (ALL historical data)...
After filtering for female players: 482 records


player_id,createdate,player_name,birthdate,club_id,club,primary_position,secondary_positions,preferred_foot,country
28478,2026-07-11,Eriko Arakawa,1979-10-30,236945,Chifure AS Elfen Saitama,F,null,null,Japan
28482,2026-07-11,Kozue Ando,1982-07-09,216779,Urawa Red Diamonds Ladies,F,null,null,Japan
29143,2026-07-11,Cristiane Rozeira,1985-05-15,248722,Flamengo,F,null,Left,Brazil
68184,2026-07-11,Lisa Klinga,null,61572,Vittsjö GIK,D,null,null,Sweden
80199,2026-07-11,Linda Sällström,1988-07-13,61572,Vittsjö GIK,M,null,null,Finland
80697,2026-07-11,Loes Geurts,1986-01-12,1889,BK Häcken FF,G,null,Right,Netherlands
94328,2026-07-11,Elin Rubensson,1993-05-11,1889,BK Häcken FF,M,null,Right,Sweden
114426,2026-07-11,Jade Bailey,1995-11-11,32871,Piteå IF,M,null,Left,Jamaica
114838,2026-07-11,Simone Boye Sørensen,1992-03-03,1884,Hammarby IF,D,null,Right,Denmark
131475,2026-07-11,Ayu Nakada,1993-08-14,320575,Omiya Ardija,M,null,null,Japan


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Deduplicate: keep only the most recent record for each player_id
window_spec = Window.partitionBy("player_id").orderBy(col("createdate").desc())

df_deduped = df.withColumn("row_num", row_number().over(window_spec)).filter(
    col("row_num") == 1
).drop("row_num", "createdate")

print(f"After deduplication: {df_deduped.count()} unique players")
display(df_deduped)

After deduplication: 482 unique players


player_id,player_name,birthdate,club_id,club,primary_position,secondary_positions,preferred_foot,country
28478,Eriko Arakawa,1979-10-30,236945,Chifure AS Elfen Saitama,F,null,null,Japan
28482,Kozue Ando,1982-07-09,216779,Urawa Red Diamonds Ladies,F,null,null,Japan
29143,Cristiane Rozeira,1985-05-15,248722,Flamengo,F,null,Left,Brazil
68184,Lisa Klinga,null,61572,Vittsjö GIK,D,null,null,Sweden
80199,Linda Sällström,1988-07-13,61572,Vittsjö GIK,M,null,null,Finland
80697,Loes Geurts,1986-01-12,1889,BK Häcken FF,G,null,Right,Netherlands
94328,Elin Rubensson,1993-05-11,1889,BK Häcken FF,M,null,Right,Sweden
114426,Jade Bailey,1995-11-11,32871,Piteå IF,M,null,Left,Jamaica
114838,Simone Boye Sørensen,1992-03-03,1884,Hammarby IF,D,null,Right,Denmark
131475,Ayu Nakada,1993-08-14,320575,Omiya Ardija,M,null,null,Japan


In [0]:
from delta.tables import DeltaTable

# Save the processed data to a new table with upsert logic
table_name = "workspace.fotmob.sofascore_player_overview_processed"

# Check if we have any data to process
row_count = df_deduped.count()
if row_count == 0:
    print(f"No new records to process in the last 7 days. Skipping write to preserve existing data.")
else:
    # If table doesn't exist, create it
    if not spark.catalog.tableExists(table_name):
        df_deduped.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Created table {table_name} with {row_count} rows")
    else:
        # Table exists - always use MERGE to upsert records
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("target").merge(
            df_deduped.alias("source"),
            "target.player_id = source.player_id"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"Merged {row_count} records into {table_name} (updated existing records or inserted new ones)")

# Display final row count
if spark.catalog.tableExists(table_name):
    final_count = spark.read.table(table_name).count()
    print(f"Total rows in {table_name}: {final_count}")

OVERWRITING workspace.fotmob.sofascore_player_overview_processed with 482 female players only...
Table workspace.fotmob.sofascore_player_overview_processed replaced with 482 rows (female players only)
Total rows in workspace.fotmob.sofascore_player_overview_processed: 482
